# 训练 + 回测模板

单品种从 all_factor 到模型训练到回测的完整流程。

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import sys
sys.path.insert(0, '/home/strat_lab')

import pandas as pd
import numpy as np
from src.config_manager import ConfigManager
from src.data_loader import DataLoader
from src.training_adapter import TrainingPipeline, build_model_folder_name
from src.backtest_adapter import BacktestPipeline

In [ ]:
# 配置
GROUP = '油脂油粕'
SYMBOL = 'A'
TRAIN_END_DATE = '2025-01-01'
TRAIN_LABEL = 5

cm = ConfigManager()
dl = DataLoader()

In [ ]:
# 加载数据
pipeline = TrainingPipeline(GROUP, SYMBOL)
df = pipeline.load_data(train_end_date=TRAIN_END_DATE, train_label=TRAIN_LABEL)
print(f"数据形状: {df.shape}")
df['pred_ret'].hist(bins=100, figsize=(8,3))

In [ ]:
# 预训练
pretrainer = pipeline.run_pretraining(df, TRAIN_END_DATE, TRAIN_LABEL)
importance = pretrainer.importance_df
importance.head(20)

In [ ]:
# TODO: 因子筛选（接入 FactorFilter）
factor_col = importance.head(300).index.tolist()
factor_col = [c for c in factor_col if c in df.columns]
print(f"筛选后因子数: {len(factor_col)}")

In [ ]:
# KFold 训练
folder_name = build_model_folder_name(SYMBOL, TRAIN_LABEL, TRAIN_END_DATE)
results = pipeline.run_training(
    df=df,
    factor_col=factor_col,
    train_end_date=TRAIN_END_DATE,
    model_folder_name=folder_name,
    n_splits=5
)

In [ ]:
# 回测
bt = BacktestPipeline(GROUP, SYMBOL)
merged = bt.run(model_folder_name=folder_name)
stats = bt.analyze(merged)
stats